# NOH0 Clean Colab Notebook — BAO/SN Ablations + Legitimate Threshold Guardrails

Run **Cells 1–10** in order. This v3 hotfix prevents generated YAMLs from being deleted immediately before `cobaya-install`. Cell 9 is the immediate diagnostic run: fixed-density BAO-only and SN-only no-H₀ ablations.

Do **not** interpret BAO-only or SN-only as a strict BAO+SN no-H₀ pass. They are diagnostic ablations to identify the residual posterior-tail driver.

Only run **Cell 11** later if we decide to attempt a legitimate same-model fixed-density P1+P2 repeat. A strict posterior-tail pass requires the weighted BAO+SN no-H₀ `q95(alpha_R) <= 0.03`, using all same-model chain files, with no local-H₀ leakage and no post-hoc threshold/prior changes.


## Cell 1 — initialize clean Colab workspace


In [ ]:
# Cell 1 — initialize clean Colab workspace
import os, pathlib, subprocess, sys, json, shutil, textwrap, zipfile, site, hashlib, platform, datetime

NOTEBOOK_VERSION = "NOH0_CLEAN_COLAB_BAO_SN_ABLATIONS_ENHANCED_v3_HOTFIX_2026-05-04"
ROOT = pathlib.Path("/content/noh0_clean_colab")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

print("Notebook version:", NOTEBOOK_VERSION)
print("Working directory:", ROOT)
print("Python:", sys.executable)
print("Platform:", platform.platform())
print("UTC:", datetime.datetime.utcnow().isoformat() + "Z")


## Cell 2 — clone the GitHub repo and select the repo state

By default this notebook uses current `main` so the committed notebook does not accidentally check out the old pre-diagnostics base commit. For a referee/release reproduction, set `PIN_REPO_COMMIT` to the final release commit or tag before running.


In [ ]:
# Cell 2 — clone the GitHub repo and pin the commit
import pathlib, subprocess, os, json

REPO_URL = "https://github.com/mike1fernand/two-perspective-edcl-validation.git"
# Default: follow current main. For a frozen reproduction, set this to the final release commit or tag.
PIN_REPO_COMMIT = None  # Set to a final release commit/tag for referee-grade reproduction.
REPO = pathlib.Path("/content/noh0_matrix_run_v4/repo")

if not REPO.exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run(["git", "fetch", "--all", "--tags"], check=False)

if PIN_REPO_COMMIT:
    print("Checking out pinned commit:", PIN_REPO_COMMIT)
    subprocess.run(["git", "checkout", "--detach", PIN_REPO_COMMIT], check=True)
else:
    print("WARNING: no pinned commit. Using current main; for referee-grade reproduction, pin to the final release commit/tag.")
    subprocess.run(["git", "checkout", "main"], check=True)
    subprocess.run(["git", "pull", "--ff-only"], check=True)

REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
REPO_STATUS = subprocess.check_output(["git", "status", "--short"], text=True).strip()
print("Repo:", REPO)
print("Commit:", REPO_COMMIT)
print("Status:", REPO_STATUS if REPO_STATUS else "clean")

if PIN_REPO_COMMIT and REPO_COMMIT != PIN_REPO_COMMIT:
    raise RuntimeError(f"Expected pinned commit {PIN_REPO_COMMIT}, got {REPO_COMMIT}")


## Cell 3 — install/check basic Python packages


In [ ]:
# Cell 3 — install/check basic Python packages
import os, site, shutil, subprocess, sys

# Keep this minimal. The repo setup script handles CLASS/classy; this cell installs Cobaya/GetDist/data tooling.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "cobaya==3.6", "getdist", "pandas", "pyyaml", "numpy", "scipy", "tabulate"
], check=True)

# Ensure console scripts are visible.
candidate_bins = ["/usr/local/bin", os.path.dirname(sys.executable)]
for p in site.getsitepackages():
    candidate_bins.append(os.path.join(os.path.dirname(p), "bin"))
for b in candidate_bins:
    if os.path.isdir(b) and b not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + b

print("cobaya-install:", shutil.which("cobaya-install"))
print("cobaya-run:", shutil.which("cobaya-run"))
subprocess.run([sys.executable, "-c", "import cobaya, getdist, pandas, yaml, numpy; print('core imports OK')"], check=True)


## Cell 4 — setup/build patched CLASS and render Tier-A YAMLs

This may take a while. It runs the repo’s Tier-A setup with MCMC skipped, then verifies that patched CLASS and the rendered no-H₀ YAML exist.


In [ ]:
# Cell 4 — setup/build patched CLASS and render Tier-A YAMLs
import os, pathlib, subprocess, sys, shutil, json

os.chdir(REPO)

WORK = pathlib.Path("/content/noh0_matrix_work_clean_v4")
LOG = pathlib.Path("/content/noh0_cell4_setup_v4.log")

if WORK.exists():
    print("Using existing WORK:", WORK)
else:
    cmd = [
        sys.executable,
        "cosmology/scripts/run_tiera1_lateonly_suite.py",
        "--profile", "iterate",
        "--work-dir", str(WORK),
        "--skip-mcmc",
        "--no-validate",
    ]
    print("Running:", " ".join(cmd))
    with LOG.open("w", encoding="utf-8", errors="replace") as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
    text = LOG.read_text(encoding="utf-8", errors="replace")
    print(text[-12000:])
    if proc.returncode != 0:
        print("Setup runner returned nonzero. Checking whether usable artifacts exist anyway.")

CLASS_PATH = WORK / "class_public"
RENDERED_YAML_DIR = WORK / "yamls"
required = [
    CLASS_PATH,
    RENDERED_YAML_DIR,
    RENDERED_YAML_DIR / "edcl_cosmo_lateonly_no_sh0es.yaml",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError("Missing required setup artifacts: " + json.dumps(missing, indent=2))

print("CLASS_PATH:", CLASS_PATH)
print("Rendered YAML dir:", RENDERED_YAML_DIR)
subprocess.run([sys.executable, "-c", "import classy; print('classy import OK')"], check=True)


## Cell 5 — create fixed-density no-H₀ base YAML by mutating the rendered repo YAML

This avoids silently drifting from the repo templates. It forces the declared Phase-1 fixed-density definition, adds derived `H0_obs`/`delta0` for reporting only, and performs structural local-H₀ leakage checks.


In [ ]:
# Cell 5 — create fixed-density no-H0 base YAML by mutating the rendered repo YAML
import pathlib, yaml, json, os, copy, hashlib

MATRIX = pathlib.Path("/content/noh0_matrix_v3")
YAML_DIR = MATRIX / "yamls"
CHAIN_DIR = MATRIX / "chains"
PKG = MATRIX / "cobaya_packages"
MANIFEST_DIR = MATRIX / "manifests"
for d in [YAML_DIR, CHAIN_DIR, PKG, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

base_rendered = pathlib.Path("/content/noh0_matrix_work_clean_v4/yamls/edcl_cosmo_lateonly_no_sh0es.yaml")
if not base_rendered.exists():
    raise FileNotFoundError(base_rendered)

base = yaml.safe_load(base_rendered.read_text(encoding="utf-8"))
if not isinstance(base, dict):
    raise RuntimeError("Rendered YAML did not parse to a dictionary.")

# Load validation config so the notebook uses the repo's declared Phase-1 fixed theory choices.
validation_config_path = pathlib.Path("cosmology/config/validation_config.yaml")
validation_config = yaml.safe_load(validation_config_path.read_text(encoding="utf-8")) if validation_config_path.exists() else {}
fixed_theory_args = (
    validation_config.get("phase1_model_definition", {})
    .get("fixed_theory_args", {})
)

# Mutate, do not rebuild from scratch.
a1b = copy.deepcopy(base)
a1b.setdefault("likelihood", {})
a1b["likelihood"] = {
    "bao.desi_dr2.desi_bao_all": None,
    "sn.pantheonplus": None,
}

a1b.setdefault("theory", {}).setdefault("classy", {})
a1b["theory"]["classy"]["path"] = str(CLASS_PATH)
a1b["theory"]["classy"].setdefault("extra_args", {})
a1b["theory"]["classy"]["extra_args"].update(fixed_theory_args)
a1b["theory"]["classy"]["extra_args"]["edcl_on"] = "yes"

params = a1b.setdefault("params", {})
params["omega_b"] = 0.02237
params["omega_cdm"] = 0.12
params.setdefault("H0", {"prior": {"min": 55.0, "max": 85.0}, "ref": 70.0, "proposal": 0.5})
params.setdefault("alpha_R", {"prior": {"min": 0.0, "max": 0.2}, "ref": 0.02, "proposal": 0.01})
params["H0_obs"] = {
    "derived": "lambda H0, alpha_R: H0 * (1.0 + alpha_R * 0.7542)",
    "latex": r"H_0^{\rm obs}",
}
params["delta0"] = {
    "derived": "lambda alpha_R: alpha_R * 0.7542",
    "latex": r"\delta_0",
}

a1b["output"] = str(CHAIN_DIR / "A1b_noH0_fixed_strict")
a1b.setdefault("sampler", {}).setdefault("mcmc", {})
m = a1b["sampler"]["mcmc"]
m.update({
    "Rminus1_stop": 0.005,
    "Rminus1_cl_stop": 0.05,
    "Rminus1_cl_level": 0.95,
    "seed": 51001,
    "max_samples": 100000,
    "learn_proposal": True,
    "learn_proposal_Rminus1_max": 30,
})
m.setdefault("covmat", "auto")
for k in ["measure_speeds", "oversample_power", "drag"]:
    m.pop(k, None)

text = yaml.safe_dump(a1b, sort_keys=False, width=1000)
forbidden = ["H0_edcl", "H0.riess2020", "riess2020", "SH0ES", "sh0es"]
checks = {
    "mutated_from_rendered_yaml": True,
    "likelihood_exact_BAO_SN": set(a1b.get("likelihood", {}).keys()) == {"bao.desi_dr2.desi_bao_all", "sn.pantheonplus"},
    "no_forbidden_local_H0_tokens": not any(tok in text for tok in forbidden),
    "has_edcl_on_yes": str(a1b.get("theory", {}).get("classy", {}).get("extra_args", {}).get("edcl_on", "")).lower() == "yes",
    "omega_b_fixed_numeric": isinstance(params.get("omega_b"), (int, float)) and abs(float(params["omega_b"]) - 0.02237) < 1e-12,
    "omega_cdm_fixed_numeric": isinstance(params.get("omega_cdm"), (int, float)) and abs(float(params["omega_cdm"]) - 0.12) < 1e-12,
    "has_H0_obs_derived_only": isinstance(params.get("H0_obs"), dict) and "derived" in params["H0_obs"],
    "has_delta0_derived_only": isinstance(params.get("delta0"), dict) and "derived" in params["delta0"],
}
for name, ok in checks.items():
    print(name, "PASS" if ok else "FAIL")
if not all(checks.values()):
    print(text[:5000])
    raise RuntimeError("A1b base YAML structural safety check failed.")

out = YAML_DIR / "A1b_noH0_fixed_strict.yaml"
out.write_text(text, encoding="utf-8")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "repo_commit": REPO_COMMIT,
    "base_rendered_yaml": str(base_rendered),
    "base_rendered_yaml_sha256": sha256(base_rendered),
    "generated_yaml": str(out),
    "generated_yaml_sha256": sha256(out),
    "checks": checks,
    "fixed_theory_args_from_validation_config": fixed_theory_args,
}
(MANIFEST_DIR / "A1b_base_yaml_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Created:", out)
print(text[:4000])


## Cell 6 — write the enhanced checkpointed diagnostic script

The script enforces weighted quantiles, multi-chain loading, ESS/tail-ESS reporting, structural no-H₀ checks, checkpoint bundles, manifests, and optional P1+P2 combined logic.


In [ ]:
# Cell 6 — write the enhanced checkpointed BAO/SN diagnostic script
import pathlib, py_compile, hashlib

script_path = pathlib.Path("/content/noh0_matrix_run_v4/repo/noh0_checkpointed_diagnostics.py")
SCRIPT = r'''#!/usr/bin/env python3
from __future__ import annotations

import argparse
import datetime as _dt
import hashlib
import json
import os
import pathlib
import platform
import shutil
import subprocess
import sys
import zipfile
from typing import Any, Iterable

import numpy as np
import pandas as pd
import yaml

MATRIX = pathlib.Path('/content/noh0_matrix_v3')
BASE_YAML_DIR = MATRIX / 'yamls'
PKG = MATRIX / 'cobaya_packages'
STRICT_Q95_THRESHOLD = 0.03
STRONG_Q95_THRESHOLD = 0.02

RUN_SPECS = {
    'C1_fixedDensity_BAO_only_noH0': {
        'likelihood_mode': 'BAO',
        'seed': 62001,
        'role': 'diagnostic_ablation_bao_only',
        'claim_scope': 'diagnostic_only_not_BAO_SN_pass_evidence',
    },
    'C2_fixedDensity_SN_only_noH0': {
        'likelihood_mode': 'SN',
        'seed': 62002,
        'role': 'diagnostic_ablation_sn_only',
        'claim_scope': 'diagnostic_only_not_BAO_SN_pass_evidence',
    },
    'P1_A1b_ultra_fixed_noH0_seed61001': {
        'likelihood_mode': 'BAO_SN',
        'seed': 61001,
        'role': 'same_model_repeat_member_P1',
        'claim_scope': 'candidate_BAO_SN_fixed_density_repeat_member',
    },
    'P2_A1b_ultra_fixed_noH0_seed61002': {
        'likelihood_mode': 'BAO_SN',
        'seed': 61002,
        'role': 'same_model_repeat_member_P2',
        'claim_scope': 'candidate_BAO_SN_fixed_density_repeat_member',
    },
}

ALLOWED_LIKELIHOODS = {
    'BAO_SN': {'bao.desi_dr2.desi_bao_all', 'sn.pantheonplus'},
    'BAO': {'bao.desi_dr2.desi_bao_all'},
    'SN': {'sn.pantheonplus'},
}

FORBIDDEN_LOCAL_H0_TOKENS = [
    'H0_edcl',
    'H0.riess2020',
    'riess2020',
    'SH0ES',
    'sh0es',
]


def now_utc() -> str:
    return _dt.datetime.now(_dt.timezone.utc).isoformat()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: pathlib.Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def safe_rel(path: pathlib.Path, root: pathlib.Path) -> str:
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)


def run_capture(cmd: list[str], cwd: pathlib.Path | None = None) -> str:
    try:
        return subprocess.check_output(cmd, cwd=str(cwd) if cwd else None, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as e:
        return f'UNAVAILABLE: {e}'


def ensure_dirs(root: pathlib.Path) -> dict[str, pathlib.Path]:
    dirs = {
        'root': root,
        'yamls': root / 'yamls',
        'chains': root / 'chains',
        'logs': root / 'logs',
        'analysis': root / 'analysis',
        'manifests': root / 'manifests',
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def run_to_log(cmd: Iterable[Any], log_path: pathlib.Path) -> int:
    cmd_s = [str(c) for c in cmd]
    print('$', ' '.join(cmd_s))
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8', errors='replace') as f:
        proc = subprocess.run(cmd_s, stdout=f, stderr=subprocess.STDOUT, text=True)
    text = log_path.read_text(encoding='utf-8', errors='replace')
    print(text[-8000:])
    return int(proc.returncode)


def load_base_a1b() -> dict[str, Any]:
    path = BASE_YAML_DIR / 'A1b_noH0_fixed_strict.yaml'
    if not path.exists():
        raise FileNotFoundError(f'Missing base fixed-density YAML: {path}. Run setup/generate cells first.')
    data = yaml.safe_load(path.read_text(encoding='utf-8'))
    if not isinstance(data, dict):
        raise RuntimeError(f'Base YAML did not parse to dict: {path}')
    return data


def deep_copy_jsonable(x: Any) -> Any:
    return json.loads(json.dumps(x))


def remove_speed_options(data: dict[str, Any]) -> None:
    m = data.get('sampler', {}).get('mcmc', {})
    if isinstance(m, dict):
        for key in ['measure_speeds', 'oversample_power', 'drag']:
            m.pop(key, None)


def force_fixed_density_params(data: dict[str, Any]) -> None:
    params = data.setdefault('params', {})
    params['omega_b'] = 0.02237
    params['omega_cdm'] = 0.12
    params.setdefault('H0', {'prior': {'min': 55.0, 'max': 85.0}, 'ref': 70.0, 'proposal': 0.5})
    params.setdefault('alpha_R', {'prior': {'min': 0.0, 'max': 0.2}, 'ref': 0.02, 'proposal': 0.01})
    params['H0_obs'] = {
        'derived': 'lambda H0, alpha_R: H0 * (1.0 + alpha_R * 0.7542)',
        'latex': r'H_0^{\rm obs}',
    }
    params['delta0'] = {
        'derived': 'lambda alpha_R: alpha_R * 0.7542',
        'latex': r'\delta_0',
    }


def set_likelihood(data: dict[str, Any], mode: str) -> None:
    if mode not in ALLOWED_LIKELIHOODS:
        raise ValueError(f'Unknown likelihood mode: {mode}')
    data['likelihood'] = {k: None for k in sorted(ALLOWED_LIKELIHOODS[mode])}


def set_mcmc(data: dict[str, Any], seed: int, max_samples: int, rminus1: float, rminus1_cl: float) -> None:
    data.setdefault('sampler', {}).setdefault('mcmc', {})
    remove_speed_options(data)
    m = data['sampler']['mcmc']
    m['seed'] = int(seed)
    m['max_samples'] = int(max_samples)
    m['Rminus1_stop'] = float(rminus1)
    m['Rminus1_cl_stop'] = float(rminus1_cl)
    m['Rminus1_cl_level'] = 0.95
    m['learn_proposal'] = True
    m['learn_proposal_Rminus1_max'] = 30
    m.setdefault('covmat', 'auto')


def flattened_strings(obj: Any) -> list[str]:
    out: list[str] = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.append(str(k))
            out.extend(flattened_strings(v))
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out.extend(flattened_strings(v))
    elif isinstance(obj, str):
        out.append(obj)
    else:
        # Numeric constants are not scanned for tokens.
        pass
    return out


def safety_check_noh0(data: dict[str, Any], run_id: str, likelihood_mode: str) -> dict[str, bool]:
    likelihood = data.get('likelihood', {})
    if not isinstance(likelihood, dict):
        raise RuntimeError(f'{run_id}: likelihood block is not a dict')
    actual_likes = set(str(k) for k in likelihood.keys())
    expected_likes = ALLOWED_LIKELIHOODS[likelihood_mode]

    strings = flattened_strings(data)
    full_text = '\n'.join(strings)
    extra_args = data.get('theory', {}).get('classy', {}).get('extra_args', {})
    params = data.get('params', {})

    checks = {
        'likelihood_exactly_expected': actual_likes == expected_likes,
        'no_forbidden_local_H0_tokens': not any(tok in full_text for tok in FORBIDDEN_LOCAL_H0_TOKENS),
        'edcl_on_yes': str(extra_args.get('edcl_on', '')).lower() == 'yes',
        'omega_b_fixed_numeric': isinstance(params.get('omega_b'), (int, float)) and abs(float(params.get('omega_b')) - 0.02237) < 1e-12,
        'omega_cdm_fixed_numeric': isinstance(params.get('omega_cdm'), (int, float)) and abs(float(params.get('omega_cdm')) - 0.12) < 1e-12,
        'alpha_R_nonnegative_prior': isinstance(params.get('alpha_R'), dict) and float(params['alpha_R'].get('prior', {}).get('min', -1)) >= 0.0,
        'has_H0_obs_derived_only': isinstance(params.get('H0_obs'), dict) and 'derived' in params['H0_obs'],
        'has_delta0_derived_only': isinstance(params.get('delta0'), dict) and 'derived' in params['delta0'],
    }

    print(f'Structural no-H0 safety checks for {run_id}:')
    print('  expected_likelihoods:', sorted(expected_likes))
    print('  actual_likelihoods:  ', sorted(actual_likes))
    for k, ok in checks.items():
        print(f'  {k}: {"PASS" if ok else "FAIL"}')
    if not all(checks.values()):
        raise RuntimeError(f'{run_id} failed structural no-H0 safety checks: ' + json.dumps(checks, indent=2))
    return checks


def write_run_yaml(dirs: dict[str, pathlib.Path], run_id: str, likelihood_mode: str, seed: int, max_samples: int, rminus1: float, rminus1_cl: float) -> pathlib.Path:
    data = deep_copy_jsonable(load_base_a1b())
    set_likelihood(data, likelihood_mode)
    force_fixed_density_params(data)
    set_mcmc(data, seed, max_samples, rminus1, rminus1_cl)
    data['output'] = str(dirs['chains'] / run_id)
    data.setdefault('debug_notes', {})['diagnostic_role'] = RUN_SPECS.get(run_id, {}).get('role', 'unknown')
    data.setdefault('debug_notes', {})['claim_scope'] = RUN_SPECS.get(run_id, {}).get('claim_scope', 'unknown')
    checks = safety_check_noh0(data, run_id, likelihood_mode)
    out = dirs['yamls'] / f'{run_id}.yaml'
    out.write_text(yaml.safe_dump(data, sort_keys=False, width=1000), encoding='utf-8')
    (dirs['manifests'] / f'{run_id}.safety_checks.json').write_text(json.dumps(checks, indent=2), encoding='utf-8')
    return out


def chain_file_candidates(prefix: str, dirs: dict[str, pathlib.Path]) -> list[pathlib.Path]:
    files = sorted(dirs['chains'].glob(prefix + '*.txt'))
    # Cobaya MCMC chains are usually <prefix>.1.txt, <prefix>.2.txt, etc.; keep only nonempty text files.
    out = [p for p in files if p.is_file() and p.stat().st_size > 0]
    return out


def load_chain_file(path: pathlib.Path) -> pd.DataFrame:
    header: list[str] | None = None
    rows: list[list[float]] = []
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith('#'):
                parts = s.lstrip('#').strip().split()
                if 'weight' in parts or 'weights' in parts or 'alpha_R' in parts or 'chi2' in parts:
                    header = parts
                continue
            if header is None:
                continue
            vals = s.split()
            if len(vals) != len(header):
                continue
            try:
                rows.append([float(x) for x in vals])
            except ValueError:
                continue
    if header is None or not rows:
        raise RuntimeError(f'Could not parse chain: {path}')
    df = pd.DataFrame(rows, columns=header)
    df['__chain_file__'] = str(path)
    return df


def load_chains(files: list[pathlib.Path]) -> pd.DataFrame:
    if not files:
        raise RuntimeError('No chain files supplied')
    dfs = [load_chain_file(p) for p in files]
    # Keep common columns plus provenance. Cobaya columns should be identical across same-model chains.
    common = set(dfs[0].columns)
    for d in dfs[1:]:
        common &= set(d.columns)
    common.add('__chain_file__')
    ordered = [c for c in dfs[0].columns if c in common]
    return pd.concat([d[ordered].copy() for d in dfs], ignore_index=True)


def weights_from_df(df: pd.DataFrame) -> np.ndarray:
    for c in ['weight', 'weights']:
        if c in df.columns:
            w = df[c].to_numpy(float)
            w[~np.isfinite(w)] = 0.0
            w[w < 0] = 0.0
            return w
    return np.ones(len(df), dtype=float)


def weighted_quantile(x: np.ndarray, q: float, w: np.ndarray) -> float | None:
    x = np.asarray(x, float)
    w = np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(w) & (w >= 0)
    x = x[ok]
    w = w[ok]
    if len(x) == 0:
        return None
    if float(np.sum(w)) <= 0.0:
        return float(np.quantile(x, q))
    order = np.argsort(x)
    xs = x[order]
    ws = w[order]
    cdf = np.cumsum(ws) / np.sum(ws)
    idx = int(np.searchsorted(cdf, q, side='left'))
    idx = min(max(idx, 0), len(xs) - 1)
    return float(xs[idx])


def ess(weights: np.ndarray) -> float:
    w = np.asarray(weights, float)
    w = w[np.isfinite(w) & (w > 0)]
    if len(w) == 0:
        return 0.0
    return float((np.sum(w) ** 2) / np.sum(w ** 2))


def chi2_column(df: pd.DataFrame) -> str | None:
    if 'chi2' in df.columns:
        return 'chi2'
    component_cols = [c for c in df.columns if c.startswith('chi2__')]
    if component_cols:
        df['chi2_total_reconstructed'] = df[component_cols].sum(axis=1)
        return 'chi2_total_reconstructed'
    return None


def summarize_dataframe(label: str, df: pd.DataFrame, chain_files: list[pathlib.Path], role: str = '') -> dict[str, Any]:
    out: dict[str, Any] = {
        'label': label,
        'status': 'parsed',
        'role': role,
        'n_chain_files': int(len(chain_files)),
        'chain_files': [str(p) for p in chain_files],
        'rows': int(len(df)),
    }
    w = weights_from_df(df)
    out['sum_weights'] = float(np.sum(w))
    out['ess'] = ess(w)
    out['ess_over_rows'] = float(out['ess'] / max(len(df), 1))

    if 'alpha_R' in df.columns:
        alpha = df['alpha_R'].to_numpy(float)
        finite = np.isfinite(alpha) & np.isfinite(w) & (w >= 0)
        alpha_f = alpha[finite]
        w_f = w[finite]
        out['q16_alpha_R'] = weighted_quantile(alpha_f, 0.16, w_f)
        out['q50_alpha_R'] = weighted_quantile(alpha_f, 0.50, w_f)
        out['q84_alpha_R'] = weighted_quantile(alpha_f, 0.84, w_f)
        out['q95_alpha_R'] = weighted_quantile(alpha_f, 0.95, w_f)
        out['unweighted_q95_alpha_R'] = float(np.quantile(alpha_f, 0.95)) if len(alpha_f) else None
        out['P_alpha_R_lt_0p03'] = float(np.sum(w_f[alpha_f < STRICT_Q95_THRESHOLD]) / np.sum(w_f)) if np.sum(w_f) > 0 else None
        out['P_alpha_R_gt_0p03'] = float(np.sum(w_f[alpha_f > STRICT_Q95_THRESHOLD]) / np.sum(w_f)) if np.sum(w_f) > 0 else None
        tail = alpha_f > STRICT_Q95_THRESHOLD
        out['tail_rows_alpha_R_gt_0p03'] = int(np.sum(tail))
        out['tail_sum_weights_alpha_R_gt_0p03'] = float(np.sum(w_f[tail]))
        out['tail_ess_alpha_R_gt_0p03'] = ess(w_f[tail]) if np.any(tail) else 0.0
        out['passes_strict_q95_0p03'] = bool(out['q95_alpha_R'] is not None and out['q95_alpha_R'] <= STRICT_Q95_THRESHOLD)
        out['passes_strong_q95_0p02'] = bool(out['q95_alpha_R'] is not None and out['q95_alpha_R'] <= STRONG_Q95_THRESHOLD)
        out['q95_minus_0p03'] = None if out['q95_alpha_R'] is None else float(out['q95_alpha_R'] - STRICT_Q95_THRESHOLD)

    ccol = chi2_column(df)
    if ccol:
        vals = df[ccol].to_numpy(float)
        if np.any(np.isfinite(vals)):
            idx = int(np.nanargmin(vals))
            best = df.iloc[idx]
            out['chi2_col'] = ccol
            out['best_chi2'] = float(best[ccol])
            for col in ['alpha_R', 'H0', 'omega_b', 'omega_cdm', 'H0_obs', 'delta0', 'chi2__BAO', 'chi2__SN', 'chi2__bao.desi_dr2.desi_bao_all', 'chi2__sn.pantheonplus']:
                if col in df.columns:
                    out['best_' + col.replace('.', '_')] = float(best[col])
    return out


def progress_tails(prefix: str, dirs: dict[str, pathlib.Path]) -> dict[str, str]:
    out: dict[str, str] = {}
    for p in sorted(dirs['chains'].glob(prefix + '*.progress')):
        lines = p.read_text(encoding='utf-8', errors='replace').splitlines()
        out[str(p)] = '\n'.join(lines[-12:])
    return out


def summarize_prefix(prefix: str, dirs: dict[str, pathlib.Path], role: str = '') -> dict[str, Any]:
    files = chain_file_candidates(prefix, dirs)
    if not files:
        return {'label': prefix, 'status': 'no_chain_found', 'role': role, 'n_chain_files': 0}
    try:
        df = load_chains(files)
        out = summarize_dataframe(prefix, df, files, role=role)
        tails = progress_tails(prefix, dirs)
        if tails:
            out['progress_tails'] = tails
        return out
    except Exception as e:
        return {'label': prefix, 'status': 'parse_failed', 'error': str(e), 'role': role, 'chain_files': [str(p) for p in files]}


def summarize_combined_p1p2(dirs: dict[str, pathlib.Path]) -> dict[str, Any]:
    prefixes = ['P1_A1b_ultra_fixed_noH0_seed61001', 'P2_A1b_ultra_fixed_noH0_seed61002']
    files: list[pathlib.Path] = []
    found_prefixes: list[str] = []
    for prefix in prefixes:
        fs = chain_file_candidates(prefix, dirs)
        if fs:
            found_prefixes.append(prefix)
            files.extend(fs)
    if len(found_prefixes) < 2:
        return {
            'label': 'COMBINED_P1P2_A1b_ultra_fixed_noH0',
            'status': 'needs_both_P1_and_P2',
            'found_prefixes': found_prefixes,
            'n_chain_files': len(files),
            'interpretation': 'Do not claim a combined same-model pass unless both P1 and P2 are present and safety-checked.',
        }
    df = load_chains(files)
    out = summarize_dataframe('COMBINED_P1P2_A1b_ultra_fixed_noH0', df, files, role='same_model_combined_P1P2_candidate_gate')
    out['found_prefixes'] = found_prefixes
    out['interpretation'] = 'This is the only candidate combined fixed-density BAO+SN no-H0 q95 gate in this notebook. BAO-only/SN-only ablations remain diagnostic only.'
    return out


def write_environment_files(dirs: dict[str, pathlib.Path]) -> None:
    freeze = run_capture([sys.executable, '-m', 'pip', 'freeze'])
    (dirs['manifests'] / 'pip_freeze.txt').write_text(freeze + '\n', encoding='utf-8')
    env = {
        'created_utc': now_utc(),
        'python': sys.version,
        'executable': sys.executable,
        'platform': platform.platform(),
        'cwd': os.getcwd(),
        'cobaya_run': shutil.which('cobaya-run'),
        'cobaya_install': shutil.which('cobaya-install'),
        'repo_commit': run_capture(['git', 'rev-parse', 'HEAD']),
        'repo_status_short': run_capture(['git', 'status', '--short']),
    }
    (dirs['manifests'] / 'environment.json').write_text(json.dumps(env, indent=2), encoding='utf-8')


def validation_config_snapshot() -> dict[str, Any]:
    p = pathlib.Path('cosmology/config/validation_config.yaml')
    if p.exists():
        try:
            return yaml.safe_load(p.read_text(encoding='utf-8'))
        except Exception as e:
            return {'error': str(e), 'path': str(p)}
    return {'error': 'validation_config.yaml not found', 'path': str(p)}


def write_manifest(dirs: dict[str, pathlib.Path], status_rows: list[dict[str, Any]], planned_run_ids: list[str]) -> None:
    files_to_hash: dict[str, str | None] = {}
    for folder_key in ['yamls', 'analysis']:
        for p in sorted(dirs[folder_key].rglob('*')):
            if p.is_file():
                files_to_hash[f'{folder_key}/{p.relative_to(dirs[folder_key])}'] = sha256_file(p)
    script_self = pathlib.Path(__file__)
    manifest = {
        'created_utc': now_utc(),
        'purpose': 'NOH0 fixed-density BAO/SN ablations and optional same-model P1/P2 q95 repeat',
        'claim_discipline': {
            'BAO_SN_q95_gate': 'Only BAO+SN no-H0 same-model chains can be used for strict posterior-tail pass/fail.',
            'BAO_only_SN_only': 'Diagnostic only; cannot establish a BAO+SN no-H0 pass.',
            'threshold': STRICT_Q95_THRESHOLD,
            'strong_threshold': STRONG_Q95_THRESHOLD,
        },
        'planned_run_ids': planned_run_ids,
        'status_rows': status_rows,
        'repo_commit': run_capture(['git', 'rev-parse', 'HEAD']),
        'repo_remote': run_capture(['git', 'remote', 'get-url', 'origin']),
        'script_path': str(script_self),
        'script_sha256': sha256_file(script_self),
        'validation_config_snapshot': validation_config_snapshot(),
        'file_hashes': files_to_hash,
    }
    (dirs['manifests'] / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')


def analyze(dirs: dict[str, pathlib.Path], run_ids: list[str], status_rows: list[dict[str, Any]]) -> None:
    summaries: list[dict[str, Any]] = []
    for r in run_ids:
        summaries.append(summarize_prefix(r, dirs, role=RUN_SPECS.get(r, {}).get('role', 'unknown')))
    combined = summarize_combined_p1p2(dirs)
    summaries.append(combined)

    out = {
        'created_utc': now_utc(),
        'status_rows': status_rows,
        'summaries': summaries,
        'interpretation_rules': {
            'strict_q95_pass': 'q95_alpha_R <= 0.03 using weighted quantiles over all chain files for the same model.',
            'strong_q95_pass': 'q95_alpha_R <= 0.02 using weighted quantiles.',
            'weighted_quantile_required': True,
            'BAO_SN_required_for_gate': True,
            'ablation_role': 'BAO/SN ablations diagnose the residual tail driver; they are not pass evidence for the BAO+SN no-H0 gate.',
            'same_model_repeat_rule': 'A claimed fixed-density posterior-tail pass should use same-model P1+P2 combined or explain why a single chain is sufficient.',
        },
    }
    (dirs['analysis'] / 'NOH0_CHECKPOINT_SUMMARY.json').write_text(json.dumps(out, indent=2), encoding='utf-8')
    table = pd.DataFrame(summaries)
    table.to_csv(dirs['analysis'] / 'NOH0_CHECKPOINT_TABLE.csv', index=False)

    report: list[str] = []
    report.append('# NOH0 checkpointed diagnostics report')
    report.append('')
    report.append('## Interpretation guardrails')
    report.append('')
    report.append(f'- Strict posterior-tail pass requires weighted `q95_alpha_R <= {STRICT_Q95_THRESHOLD}` on the BAO+SN no-H0 model.')
    report.append('- BAO-only and SN-only are diagnostic ablations only; they cannot by themselves establish a BAO+SN no-H0 pass.')
    report.append('- A same-model P1/P2 combined result is reported when both repeat chains are present.')
    report.append('- A pass must not be obtained by changing the threshold, tightening the alpha prior after seeing results, or selecting a favourable seed.')
    report.append('')
    report.append('## Summary table')
    report.append('')
    try:
        report.append(table.to_markdown(index=False))
    except Exception:
        report.append(table.to_string(index=False))
    report.append('')
    report.append('## Current gate logic')
    report.append('')
    for s in summaries:
        label = s.get('label')
        status = s.get('status')
        q95 = s.get('q95_alpha_R')
        pass_gate = s.get('passes_strict_q95_0p03')
        role = s.get('role', '')
        report.append(f'- `{label}` [{role}]: status={status}, q95={q95}, strict_q95_pass={pass_gate}')
    (dirs['analysis'] / 'NOH0_CHECKPOINT_REPORT.md').write_text('\n'.join(report) + '\n', encoding='utf-8')

    write_manifest(dirs, status_rows, run_ids)


def bundle(dirs: dict[str, pathlib.Path]) -> pathlib.Path:
    out = dirs['root'] / 'NOH0_CHECKPOINT_UPLOAD.zip'
    if out.exists():
        out.unlink()
    include = [('yamls', 'yamls'), ('chains', 'chains'), ('logs', 'logs'), ('analysis', 'analysis'), ('manifests', 'manifests')]
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for folder_key, arc in include:
            for p in dirs[folder_key].rglob('*'):
                if p.is_file():
                    z.write(p, f'{arc}/{p.relative_to(dirs[folder_key])}')
    return out


def delete_existing_run_files(dirs: dict[str, pathlib.Path], run_id: str) -> None:
    # v3 hotfix: do NOT delete generated YAMLs here. write_run_yaml() writes
    # <run_id>.yaml before run_one() is called; deleting yamls at this point
    # caused cobaya-install to fail with FileNotFoundError. Chain/log/analysis
    # cleanup is still safe. The YAML is overwritten by write_run_yaml() for
    # each planned run, so preserving it here does not leave stale config.
    for folder_key in ['chains', 'logs', 'analysis']:
        for p in dirs[folder_key].glob(run_id + '*'):
            if p.is_file():
                p.unlink()
            elif p.is_dir():
                shutil.rmtree(p)


def run_one(dirs: dict[str, pathlib.Path], run_id: str, ypath: pathlib.Path, skip_existing: bool) -> dict[str, Any]:
    existing = chain_file_candidates(run_id, dirs)
    if existing and skip_existing:
        print(f'Skipping existing {run_id}: {[str(p) for p in existing]}')
        return {'run_id': run_id, 'status': 'skipped_existing', 'chains': [str(p) for p in existing], 'yaml': str(ypath)}

    delete_existing_run_files(dirs, run_id)
    if not ypath.exists():
        raise FileNotFoundError(f'{run_id}: generated YAML disappeared before install: {ypath}')
    row: dict[str, Any] = {'run_id': run_id, 'yaml': str(ypath), 'started_utc': now_utc()}

    rc = run_to_log(['cobaya-install', ypath, '-p', PKG], dirs['logs'] / f'{run_id}.install.log')
    row['install_rc'] = rc
    if rc != 0:
        row['status'] = 'install_failed'
        row['finished_utc'] = now_utc()
        return row

    rc = run_to_log(['cobaya-run', ypath, '--test', '-p', PKG], dirs['logs'] / f'{run_id}.test.log')
    row['test_rc'] = rc
    if rc != 0:
        row['status'] = 'test_failed'
        row['finished_utc'] = now_utc()
        return row

    rc = run_to_log(['cobaya-run', ypath, '-p', PKG], dirs['logs'] / f'{run_id}.run.log')
    row['run_rc'] = rc
    row['status'] = 'passed' if rc == 0 else 'run_failed'
    chains = chain_file_candidates(run_id, dirs)
    if chains:
        row['chains'] = [str(p) for p in chains]
    row['finished_utc'] = now_utc()
    return row



def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument('--mode', choices=['ablations_first', 'p2_only', 'p1p2', 'all'], default='ablations_first')
    parser.add_argument('--output-root', default='/content/noh0_checkpointed')
    parser.add_argument('--skip-existing', action='store_true')
    parser.add_argument('--ablation-max-samples', type=int, default=80000)
    parser.add_argument('--p-repeat-max-samples', type=int, default=150000)
    parser.add_argument('--p2-max-samples', type=int, default=None, help='Deprecated alias; overrides --p-repeat-max-samples if supplied.')
    args = parser.parse_args()
    if args.p2_max_samples is not None:
        args.p_repeat_max_samples = args.p2_max_samples

    dirs = ensure_dirs(pathlib.Path(args.output_root))
    write_environment_files(dirs)
    print('Output root:', dirs['root'])
    print('Mode:', args.mode)
    print('Repo commit:', run_capture(['git', 'rev-parse', 'HEAD']))

    planned_specs: list[tuple[str, str, int, int, float, float]] = []
    if args.mode in ['ablations_first', 'all']:
        planned_specs.append(('C1_fixedDensity_BAO_only_noH0', 'BAO', 62001, args.ablation_max_samples, 0.005, 0.05))
        planned_specs.append(('C2_fixedDensity_SN_only_noH0', 'SN', 62002, args.ablation_max_samples, 0.005, 0.05))
    if args.mode in ['p1p2', 'all']:
        planned_specs.append(('P1_A1b_ultra_fixed_noH0_seed61001', 'BAO_SN', 61001, args.p_repeat_max_samples, 0.0015, 0.02))
        planned_specs.append(('P2_A1b_ultra_fixed_noH0_seed61002', 'BAO_SN', 61002, args.p_repeat_max_samples, 0.0015, 0.02))
    if args.mode == 'p2_only':
        planned_specs.append(('P2_A1b_ultra_fixed_noH0_seed61002', 'BAO_SN', 61002, args.p_repeat_max_samples, 0.0015, 0.02))

    planned: list[tuple[str, pathlib.Path]] = []
    for run_id, like_mode, seed, max_samples, r1, r1cl in planned_specs:
        ypath = write_run_yaml(dirs, run_id, like_mode, seed, max_samples, r1, r1cl)
        planned.append((run_id, ypath))

    run_ids = [x[0] for x in planned]
    status_rows: list[dict[str, Any]] = []
    analyze(dirs, run_ids, status_rows)
    out = bundle(dirs)
    print('Initial manifest/checkpoint bundle:', out)

    for run_id, ypath in planned:
        print('\n' + '=' * 88)
        print('RUNNING', run_id)
        print('=' * 88)
        try:
            row = run_one(dirs, run_id, ypath, args.skip_existing)
        except Exception as e:
            row = {'run_id': run_id, 'status': 'exception', 'error': repr(e), 'yaml': str(ypath), 'finished_utc': now_utc()}
        status_rows.append(row)
        analyze(dirs, run_ids, status_rows)
        out = bundle(dirs)
        print('Checkpoint bundle:', out)
        print('Size MB:', out.stat().st_size / 1024 / 1024)

    analyze(dirs, run_ids, status_rows)
    out = bundle(dirs)
    print('\nFinal checkpoint bundle:')
    print(out)
    print('Size MB:', out.stat().st_size / 1024 / 1024)
    print('Upload this file here.')
    return 0


if __name__ == '__main__':
    raise SystemExit(main())
'''
script_path.write_text(SCRIPT, encoding="utf-8")
py_compile.compile(str(script_path), doraise=True)
sha = hashlib.sha256(script_path.read_bytes()).hexdigest()
print("Wrote and syntax-checked:", script_path)
print("Script SHA256:", sha)
print("Script lines:", len(SCRIPT.splitlines()))


## Cell 7 — final environment and safety sanity check before running


In [ ]:
# Cell 7 — final environment and safety sanity check before running BAO/SN ablations
import shutil, subprocess, sys, pathlib, os, json, yaml

print("cobaya-install:", shutil.which("cobaya-install"))
print("cobaya-run:", shutil.which("cobaya-run"))
if shutil.which("cobaya-install") is None or shutil.which("cobaya-run") is None:
    raise RuntimeError("Cobaya CLI commands are missing. Re-run Cell 3.")

subprocess.run([sys.executable, "-c", "import cobaya, classy, pandas, yaml, numpy; print('cobaya/classy/data imports OK')"], check=True)

base_yaml = pathlib.Path("/content/noh0_matrix_v3/yamls/A1b_noH0_fixed_strict.yaml")
script = pathlib.Path("/content/noh0_matrix_run_v4/repo/noh0_checkpointed_diagnostics.py")
print("A1b base YAML exists:", base_yaml.exists(), base_yaml)
print("Diagnostic script exists:", script.exists(), script)
if not base_yaml.exists():
    raise RuntimeError("Missing A1b base YAML. Re-run Cell 5.")
if not script.exists():
    raise RuntimeError("Missing diagnostic script. Re-run Cell 6.")

base = yaml.safe_load(base_yaml.read_text(encoding="utf-8"))
print("Likelihoods:", list(base.get("likelihood", {}).keys()))
if set(base.get("likelihood", {}).keys()) != {"bao.desi_dr2.desi_bao_all", "sn.pantheonplus"}:
    raise RuntimeError("Base YAML does not have the expected BAO+SN no-H0 likelihood block.")
print("Ready to run Cell 9.")


## Cell 8 — optional but recommended: mount Google Drive

Run this so checkpoint ZIPs survive if Colab resets. If you skip it, Cell 9 saves to `/content/noh0_checkpointed`.


In [ ]:
# Cell 8 — optional but recommended: mount Google Drive for persistent checkpoint output
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted. Diagnostics will save to /content/drive/MyDrive/noh0_checkpointed")


## Cell 9 — RUN NOW: BAO-only and SN-only fixed-density no-H₀ ablations

This is the immediate next run. It should create/update `NOH0_CHECKPOINT_UPLOAD.zip` after each completed run.


In [ ]:
# Cell 9 — RUN NOW: fixed-density BAO-only and SN-only no-H0 ablations
import pathlib, subprocess, sys, os

os.chdir(REPO)
script = pathlib.Path("/content/noh0_matrix_run_v4/repo/noh0_checkpointed_diagnostics.py")

output_root = "/content/drive/MyDrive/noh0_checkpointed"
if not pathlib.Path("/content/drive/MyDrive").exists():
    output_root = "/content/noh0_checkpointed"
    print("Drive not found; using temporary output root:", output_root)
else:
    print("Using Drive output root:", output_root)

subprocess.run(
    [
        sys.executable,
        str(script),
        "--mode", "ablations_first",
        "--output-root", output_root,
        "--skip-existing",
    ],
    check=True,
)

print("Finished BAO/SN diagnostic ablations.")
print("Upload this file:")
print(str(pathlib.Path(output_root) / "NOH0_CHECKPOINT_UPLOAD.zip"))


## Cell 10 — inspect/check output bundle

After Cell 9, download/upload `NOH0_CHECKPOINT_UPLOAD.zip`.


In [ ]:
# Cell 10 — inspect and verify diagnostic output bundle
import pathlib, zipfile, json, pandas as pd

output_root = pathlib.Path("/content/drive/MyDrive/noh0_checkpointed")
if not output_root.exists():
    output_root = pathlib.Path("/content/noh0_checkpointed")

bundle = output_root / "NOH0_CHECKPOINT_UPLOAD.zip"
print("Bundle:", bundle)
print("Exists:", bundle.exists())
if not bundle.exists():
    raise RuntimeError("Bundle not found. Re-run Cell 9 or check the output root.")

print("Size MB:", bundle.stat().st_size / 1024 / 1024)
with zipfile.ZipFile(bundle) as z:
    names = z.namelist()
    print("Files in bundle:")
    for n in names[:120]:
        print("-", n)
    required_prefixes = ["analysis/NOH0_CHECKPOINT_SUMMARY.json", "analysis/NOH0_CHECKPOINT_REPORT.md", "manifests/MANIFEST.json"]
    missing = [r for r in required_prefixes if r not in names]
    if missing:
        raise RuntimeError("Bundle missing required files: " + str(missing))

report = output_root / "analysis" / "NOH0_CHECKPOINT_REPORT.md"
summary = output_root / "analysis" / "NOH0_CHECKPOINT_SUMMARY.json"
if report.exists():
    print("\n===== REPORT =====")
    print(report.read_text(encoding="utf-8")[:12000])
if summary.exists():
    data = json.loads(summary.read_text(encoding="utf-8"))
    print("\nSummary labels:", [s.get("label") for s in data.get("summaries", [])])


## Cell 11 — OPTIONAL LATER ONLY: same-model P1+P2 fixed-density BAO+SN no-H₀ repeat

Do **not** run this until the ablation output has been analyzed. This cell is the legitimate posterior-tail pass attempt: same model, two seeds, weighted combined q95, no threshold changes.


In [ ]:
# Cell 11 — OPTIONAL LATER ONLY: same-model P1+P2 fixed-density BAO+SN no-H0 repeat
# Do not run this until the Cell 9 ablation zip has been analyzed.

import pathlib, subprocess, sys, os

os.chdir(REPO)
script = pathlib.Path("/content/noh0_matrix_run_v4/repo/noh0_checkpointed_diagnostics.py")

output_root = "/content/drive/MyDrive/noh0_checkpointed"
if not pathlib.Path("/content/drive/MyDrive").exists():
    output_root = "/content/noh0_checkpointed"

subprocess.run(
    [
        sys.executable,
        str(script),
        "--mode", "p1p2",
        "--output-root", output_root,
        "--skip-existing",
        "--p-repeat-max-samples", "150000",
    ],
    check=True,
)

print("Finished optional P1+P2 same-model repeat/combined analysis.")
print("Upload this file:")
print(str(pathlib.Path(output_root) / "NOH0_CHECKPOINT_UPLOAD.zip"))
